In [1]:
!pip install timm albumentations --quiet

In [ ]:
import os
import numpy as np
import pandas as pd
import random
import cv2
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

print("PyTorch:", torch.__version__)
print("Timm:   ", timm.__version__)
print("GPU:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [3]:
CONFIG = {
    "img_size"   : 224,
    "batch_size" : 24,
    "epochs"     : 25 ,
    "lr"         : 2e-4,
    "num_classes": 13,
    "seed"       : 42,
    "num_workers": 2,
}
CACHE_DIR       = "/kaggle/input/datasets/ellefiala/wbc-cache/preprocessed"
BEST_MODEL_CNX  = "/kaggle/working/best_model_convnext.pth"
BEST_MODEL_EFF  = "/kaggle/working/best_model_effv2.pth"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Classes grouped by problem type
CRITICAL_CLASSES = ["PLY", "PC", "PMY"]
SEVERE_CLASSES   = ["MMY", "VLY", "BNE", "BA", "MY"]
CONFUSED_PAIRS   = [("BNE","MMY"), ("MMY","MY"), ("LY","VLY"), ("PLY","LY"), ("PC","LY")]

print("Config ready.")

Config ready.


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
print("Seed set.")

In [ ]:
base_path       = "/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge"
train_path      = os.path.join(base_path, "train_metadata.csv")
test_path       = os.path.join(base_path, "test_metadata.csv")
sample_sub_path = os.path.join(base_path, "sample_submission.csv")
train_images    = os.path.join(base_path, "train")
test_images     = os.path.join(base_path, "test")

for path in [train_path, test_path, sample_sub_path, train_images, test_images]:
    print(f"{'OK' if os.path.exists(path) else 'MISSING'} {path}")

In [ ]:
train_df   = pd.read_csv(train_path)
test_df    = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

classes  = sorted(train_df["label"].unique())
label2id = {label: i for i, label in enumerate(classes)}
id2label = {i: label for label, i in label2id.items()}
train_df["label_id"] = train_df["label"].map(label2id)

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)
print("\nLabel mapping:")
for label, idx in label2id.items():
    count = (train_df["label"] == label).sum()
    flag  = " CRITICAL" if label in CRITICAL_CLASSES else \
            " SEVERE"   if label in SEVERE_CLASSES   else ""
    print(f"  {idx:2d} — {label:4s} — {count:5d} images{flag}")

In [ ]:
import os
cache_files = os.listdir(CACHE_DIR)
print(f"Files in cache: {len(cache_files)}")
print(f"Example: {cache_files[0] if cache_files else 'EMPTY'}")
print(f"Train example ID: {train_df['ID'].iloc[0]}")

In [ ]:
counts = train_df["label"].value_counts()
colors = ["red" if c in CRITICAL_CLASSES else
          "orange" if c in SEVERE_CLASSES else
          "steelblue" for c in counts.index]

plt.figure(figsize=(13, 4))
bars = plt.bar(counts.index, counts.values, color=colors)
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 30, str(val),
             ha="center", fontsize=8)
plt.title("Class Distribution — Red=Critical, Orange=Severe, Blue=OK")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("/kaggle/working/class_distribution.png", dpi=150)
plt.show()

In [ ]:
def get_noise_level(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if lap_var > 10000:
        return "extreme"
    elif lap_var > 2000:
        return "medium"
    else:
        return "clean"

def denoise_image(image):
    level = get_noise_level(image)
    if level == "extreme":
        # Strong denoising for heavily corrupted images
        denoised = cv2.fastNlMeansDenoisingColored(
            image, None, h=40, hColor=40,
            templateWindowSize=7, searchWindowSize=21
        )
        return denoised
    elif level == "medium":
        return cv2.fastNlMeansDenoisingColored(
            image, None, h=10, hColor=10,
            templateWindowSize=7, searchWindowSize=21
        )
    else:
        return image

def normalize_stain(image):
    image_float = image.astype(np.float32) / 255.0
    lab = cv2.cvtColor(image_float, cv2.COLOR_RGB2LAB)
    target_mean = np.array([70.0,  5.0, -10.0])
    target_std  = np.array([15.0,  8.0,   8.0])
    for i in range(3):
        ch = lab[:, :, i]
        ch = (ch - ch.mean()) / (ch.std() + 1e-6)
        ch = ch * target_std[i] + target_mean[i]
        lab[:, :, i] = ch
    normalized = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return np.clip(normalized * 255, 0, 255).astype(np.uint8)

def apply_clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

def is_noisy(image, threshold=2000):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var() > threshold

def preprocess_image(image):
    image = denoise_image(image)
    image = normalize_stain(image)
    image = apply_clahe(image)
    return image

print("Preprocessing functions updated.")

In [ ]:
def show_denoising_effect(class_name, n_noisy=3):
    """Show before/after denoising on noisy images of a class."""
    df_cls  = train_df[train_df["label"] == class_name]
    noisy_rows = []
    
    for _, row in df_cls.iterrows():
        img = np.array(Image.open(
            os.path.join(train_images, row["ID"])).convert("RGB"))
        if is_noisy(img):
            noisy_rows.append((row, img))
        if len(noisy_rows) >= n_noisy:
            break
    
    if not noisy_rows:
        print(f"No noisy images found for {class_name}")
        return
    
    fig, axes = plt.subplots(len(noisy_rows), 4, figsize=(16, 4*len(noisy_rows)))
    if len(noisy_rows) == 1:
        axes = [axes]
    
    for i, (row, img) in enumerate(noisy_rows):
        denoised   = denoise_image(img)
        normalized = normalize_stain(denoised)
        final      = apply_clahe(normalized)
        
        axes[i][0].imshow(img)
        axes[i][0].set_title("Original (noisy)")
        axes[i][0].axis("off")
        
        axes[i][1].imshow(denoised)
        axes[i][1].set_title("After Denoising")
        axes[i][1].axis("off")
        
        axes[i][2].imshow(normalized)
        axes[i][2].set_title("After Stain Norm")
        axes[i][2].axis("off")
        
        axes[i][3].imshow(final)
        axes[i][3].set_title("After CLAHE")
        axes[i][3].axis("off")
    
    plt.suptitle(f"{class_name} — Denoising Pipeline", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/denoise_{class_name}.png", dpi=150)
    plt.show()

# Test on most affected classes
for cls in ["BNE", "VLY", "LY", "MMY"]:
    show_denoising_effect(cls)

In [ ]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=CONFIG["seed"]
)
train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
print("\nVal class distribution:")
for label, idx in label2id.items():
    print(f"  {label:4s} — {(val_data['label']==label).sum():4d}")

In [ ]:
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(CONFIG["num_classes"]),
    y=train_data["label"].map(label2id).values
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).cuda()

print("Class weights:")
for label, idx in label2id.items():
    print(f"  {label:4s} — count: {(train_data['label']==label).sum():5d} | weight: {class_weights[idx]:.4f}")

In [ ]:
train_transforms = A.Compose([
    A.Resize(CONFIG["img_size"], CONFIG["img_size"]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=45, p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.4),
    A.RandomResizedCrop(
        size=(CONFIG["img_size"], CONFIG["img_size"]),
        scale=(0.8, 1.0), ratio=(0.9, 1.1), p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(CONFIG["img_size"], CONFIG["img_size"]),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

tta_transforms = A.Compose([
    A.Resize(CONFIG["img_size"], CONFIG["img_size"]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

print("Transforms defined.")

In [ ]:
class WBCDataset(Dataset):
    def __init__(self, df, image_dir, transforms=None, has_labels=True):
        self.df         = df.reset_index(drop=True)
        self.image_dir  = image_dir
        self.transforms = transforms
        self.has_labels = has_labels
        self.cached     = set(os.listdir(CACHE_DIR)) if os.path.exists(CACHE_DIR) else set()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if row["ID"] in self.cached:
            image = np.array(Image.open(
                os.path.join(CACHE_DIR, row["ID"])).convert("RGB"))
        else:
            image = np.array(Image.open(
                os.path.join(self.image_dir, row["ID"])).convert("RGB"))
            image = preprocess_image(image)
        if self.transforms:
            image = self.transforms(image=image)["image"]
        if self.has_labels:
            return image, label2id[row["label"]]
        return image

_ds = WBCDataset(train_data, train_images, val_transforms)
_img, _lbl = _ds[0]
print(f"Image shape: {_img.shape} | Label: {_lbl} ({id2label[_lbl]})")

In [ ]:
train_dataset = WBCDataset(train_data, train_images, train_transforms)
val_dataset   = WBCDataset(val_data,   train_images, val_transforms)

sample_weights = np.array([
    1.0 / np.sqrt((train_data["label"] == label).sum())
    for label in train_data["label"].values
])
sample_weights = torch.tensor(sample_weights, dtype=torch.float)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    sampler=sampler,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
print("Counting class frequencies after sqrt sampling...")
sampled_labels = []
for _, labels in tqdm(train_loader, desc="Counting", leave=False):
    sampled_labels.extend(labels.numpy())

sampled_counts  = Counter(sampled_labels)
original_counts = train_data["label"].map(label2id).value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
ax1.bar(classes,
        [original_counts.get(i, 0) for i in range(len(classes))],
        color="steelblue")
ax1.set_title("Original Distribution")
ax1.set_xticklabels(classes, rotation=30)

sampled_vals = [sampled_counts.get(i, 0) for i in range(len(classes))]
ax2.bar(classes, sampled_vals, color="darkorange")
ax2.set_title("After Sqrt Sampling")
ax2.set_xticklabels(classes, rotation=30)

plt.tight_layout()
plt.savefig("/kaggle/working/sampling_balance.png", dpi=150)
plt.show()

print("\nOriginal vs Sampled:")
for i, label in enumerate(classes):
    orig = original_counts.get(i, 0)
    samp = sampled_counts.get(i, 0)
    print(f"  {label:4s} — original: {orig:5d} | sampled: {samp:5d}")

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=1.5):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets,
                             weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features = F.normalize(features, dim=1)
        sim      = torch.matmul(features, features.T) / self.temperature

        # Numerical stability — subtract row-wise max before exp
        sim = sim - sim.max(dim=1, keepdim=True)[0].detach()

        labels  = labels.view(-1, 1)
        mask    = torch.eq(labels, labels.T).float().to(features.device)
        eye     = torch.eye(len(labels)).to(features.device)
        mask    = mask - eye

        exp_sim  = torch.exp(sim) * (1 - eye)
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        loss     = -(mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-8)
        return loss.mean()


focal_loss  = FocalLoss(weight=class_weights, gamma=1.5)
supcon_loss = SupConLoss(temperature=0.07)
print("Losses defined.")

In [ ]:
def cutmix_batch(images, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(images.size(0)).to(images.device)
    W, H = images.size(2), images.size(3)
    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = max(0, cx - cut_w // 2)
    y1 = max(0, cy - cut_h // 2)
    x2 = min(W, cx + cut_w // 2)
    y2 = min(H, cy + cut_h // 2)
    mixed = images.clone()
    mixed[:, :, x1:x2, y1:y2] = images[idx, :, x1:x2, y1:y2]
    lam = 1 - (x2-x1)*(y2-y1) / (W*H)
    return mixed, labels, labels[idx], lam


def train_one_epoch(model, loader, focal_loss, supcon_loss,
                    optimizer, scheduler, supcon_weight=0.03,
                    cutmix_prob=0.5):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images, labels = images.cuda(), labels.cuda()
        optimizer.zero_grad()

        use_cutmix = random.random() < cutmix_prob
        if use_cutmix:
            images, labels_a, labels_b, lam = cutmix_batch(images, labels)
            logits, _ = model(images, return_features=True)
            loss = (lam * focal_loss(logits, labels_a) +
                    (1 - lam) * focal_loss(logits, labels_b))
        else:
            logits, proj = model(images, return_features=True)
            loss = focal_loss(logits, labels)
            loss = loss + supcon_weight * supcon_loss(proj, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / len(loader), f1


def validate(model, loader, focal_loss):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val", leave=False):
            images, labels = images.cuda(), labels.cuda()
            logits  = model(images)  # no features needed
            loss    = focal_loss(logits, labels)
            total_loss += loss.item()
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / len(loader), f1, all_labels, all_preds


def run_training(model, train_loader, val_loader,
                 focal_loss, supcon_loss,
                 optimizer, scheduler,
                 n_epochs, start_epoch,
                 best_f1, best_model_path):
    history = {"train_loss": [], "train_f1": [],
               "val_loss":   [], "val_f1":   []}

    for epoch in range(start_epoch, n_epochs):
        print(f"\nEpoch {epoch+1}/{n_epochs} " + "-"*30)
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, focal_loss, supcon_loss,
            optimizer, scheduler,
            supcon_weight=0.03, cutmix_prob=0.5
        )
        val_loss, val_f1, _, _ = validate(model, val_loader, focal_loss)
        # DO NOT call scheduler.step() here — already called per batch

        lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(train_loss)
        history["train_f1"].append(train_f1)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)

        print(f"  Train  loss={train_loss:.4f}  f1={train_f1:.4f}")
        print(f"  Val    loss={val_loss:.4f}  f1={val_f1:.4f}  lr={lr:.2e}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"  *** Best saved — Val F1: {best_f1:.4f}")

    print(f"\nDone. Best Val F1: {best_f1:.4f}")
    return best_f1, history

print("Training functions defined.")

In [ ]:
class WBCModel(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=True,
            num_classes=0, global_pool="avg"
        )
        in_features = self.backbone.num_features
        self.projector = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(dropout),
            nn.Linear(in_features, 768),
            nn.GELU(),
            nn.BatchNorm1d(768),
            nn.Dropout(dropout * 0.5),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout * 0.25),
            nn.Linear(256, num_classes)
        )

    def forward(self, x, return_features=False):
        features = self.backbone(x)
        logits   = self.head(features)
        if return_features:
            return logits, self.projector(features)
        return logits

model = WBCModel("convnext_small.fb_in22k_ft_in1k",
                  CONFIG["num_classes"]).cuda()
total_p = sum(p.numel() for p in model.parameters()) / 1e6
print(f"ConvNeXt-Small — {total_p:.1f}M params")

In [ ]:
optimizer_cnx = optim.AdamW(
    model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)

scheduler_cnx = optim.lr_scheduler.OneCycleLR(
    optimizer_cnx,
    max_lr=CONFIG["lr"],
    steps_per_epoch=len(train_loader),
    epochs=CONFIG["epochs"],
    pct_start=0.1,
    anneal_strategy="cos",
    div_factor=25,
    final_div_factor=1e4
)

best_f1_cnx, history_cnx = run_training(
    model, train_loader, val_loader,
    focal_loss, supcon_loss,
    optimizer_cnx, scheduler_cnx,
    n_epochs=CONFIG["epochs"], start_epoch=0,
    best_f1=0.0,
    best_model_path=BEST_MODEL_CNX
)

In [ ]:
def plot_history(history, title, start_epoch=0):
    epochs = range(start_epoch+1,
                   start_epoch+len(history["val_f1"])+1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, history["train_loss"],
             label="Train Loss", marker="o", markersize=3)
    ax1.plot(epochs, history["val_loss"],
             label="Val Loss",   marker="o", markersize=3)
    ax1.set_title(f"{title} — Loss")
    ax1.set_xlabel("Epoch")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2.plot(epochs, history["train_f1"],
             label="Train F1", marker="o", markersize=3)
    ax2.plot(epochs, history["val_f1"],
             label="Val F1",   marker="o", markersize=3)
    ax2.axhline(y=max(history["val_f1"]),
                color="red", linestyle="--", alpha=0.5,
                label=f"Best={max(history['val_f1']):.4f}")
    ax2.set_title(f"{title} — Macro F1")
    ax2.set_xlabel("Epoch")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.suptitle(f"Best Val F1: {max(history['val_f1']):.4f}",
                 fontsize=12)
    plt.tight_layout()
    plt.savefig(
        f"/kaggle/working/{title.replace(' ','_')}_curves.png",
        dpi=150)
    plt.show()

plot_history(history_cnx, "ConvNeXt-Base", start_epoch=0)

In [ ]:
model_eff = WBCModel(
    "tf_efficientnetv2_s.in21k_ft_in1k",
    CONFIG["num_classes"]).cuda()
total_p = sum(p.numel() for p in model_eff.parameters()) / 1e6
print(f"EfficientNetV2-S — {total_p:.1f}M params")

optimizer_eff = optim.AdamW(
    model_eff.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)

scheduler_eff = optim.lr_scheduler.OneCycleLR(
    optimizer_eff,
    max_lr=CONFIG["lr"],
    steps_per_epoch=len(train_loader),
    epochs=CONFIG["epochs"],
    pct_start=0.1,
    anneal_strategy="cos",
    div_factor=25,
    final_div_factor=1e4
)

best_f1_eff, history_eff = run_training(
    model_eff, train_loader, val_loader,
    focal_loss, supcon_loss,
    optimizer_eff, scheduler_eff,
    n_epochs=CONFIG["epochs"], start_epoch=0,
    best_f1=0.0,
    best_model_path=BEST_MODEL_EFF
)

In [ ]:
plot_history(history_eff, "EfficientNetV2-S", start_epoch=0)

In [ ]:
def get_val_preds(model, model_path, loader):
    model.load_state_dict(torch.load(model_path))
    _, _, all_labels, all_preds = validate(
        model, loader, focal_loss)
    return all_labels, all_preds

labels_cnx, preds_cnx = get_val_preds(
    model,     BEST_MODEL_CNX, val_loader)
labels_eff, preds_eff = get_val_preds(
    model_eff, BEST_MODEL_EFF, val_loader)

f1_cnx = f1_score(labels_cnx, preds_cnx, average=None, zero_division=0)
f1_eff = f1_score(labels_eff, preds_eff, average=None, zero_division=0)

x     = np.arange(len(classes))
width = 0.35
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x-width/2, f1_cnx, width,
       label=f"ConvNeXt-Base (F1={best_f1_cnx:.4f})",
       color="steelblue")
ax.bar(x+width/2, f1_eff, width,
       label=f"EffNetV2-S (F1={best_f1_eff:.4f})",
       color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylabel("F1 Score")
ax.set_title("Per-class F1 — ConvNeXt-Base vs EfficientNetV2-S")
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/per_class_f1.png", dpi=150)
plt.show()

print("\nConvNeXt-Base — Classification Report:")
print(classification_report(
    labels_cnx, preds_cnx, target_names=classes, zero_division=0))
print("\nEfficientNetV2-S — Classification Report:")
print(classification_report(
    labels_eff, preds_eff, target_names=classes, zero_division=0))

In [ ]:
def plot_confusion_matrix(labels, preds, title):
    cm      = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(11, 9))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=classes, yticklabels=classes)
    plt.title(f"{title} — Confusion Matrix (normalized)")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(
        f"/kaggle/working/{title.replace(' ','_')}_confusion.png",
        dpi=150)
    plt.show()

plot_confusion_matrix(labels_cnx, preds_cnx, "ConvNeXt-Base")
plot_confusion_matrix(labels_eff, preds_eff, "EfficientNetV2-S")

In [ ]:
def predict_tta(model, test_df, image_dir, n_tta=7):
    model.eval()
    all_probs = []
    for i in range(n_tta):
        aug = val_transforms if i == 0 else tta_transforms
        ds  = WBCDataset(test_df, image_dir, aug,
                         has_labels=False)
        dl  = DataLoader(ds, batch_size=48, shuffle=False,
                         num_workers=CONFIG["num_workers"],
                         pin_memory=True)
        probs = []
        with torch.no_grad():
            for images in tqdm(dl, desc=f"TTA {i+1}/{n_tta}",
                               leave=False):
                out = model(images.cuda())
                probs.append(
                    torch.softmax(out, dim=1).cpu().numpy())
        all_probs.append(np.concatenate(probs, axis=0))
    return np.mean(all_probs, axis=0)

print("TTA function defined.")

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_CNX))
model_eff.load_state_dict(torch.load(BEST_MODEL_EFF))

print("Running TTA — ConvNeXt-Base...")
probs_cnx = predict_tta(model,     test_df, test_images, n_tta=7)

print("Running TTA — EfficientNetV2-S...")
probs_eff = predict_tta(model_eff, test_df, test_images, n_tta=7)

w_cnx    = best_f1_cnx
w_eff    = best_f1_eff
ensemble = (w_cnx * probs_cnx + w_eff * probs_eff) / (w_cnx + w_eff)

pred_ids    = ensemble.argmax(axis=1)
pred_labels = [id2label[i] for i in pred_ids]

print(f"\nWeights — ConvNeXt: {w_cnx:.4f} | EffNetV2: {w_eff:.4f}")
print("Prediction distribution:")
print(pd.Series(pred_labels).value_counts())

In [ ]:
sub_out = sample_sub[["Unnamed: 0", "ID"]].copy()
sub_out["label"] = pred_labels
sub_out.to_csv(SUBMISSION_PATH, index=False)

assert len(sub_out) == len(test_df), "Row count mismatch!"
assert set(sub_out["label"].unique()).issubset(set(classes)), "Unknown labels!"
print(f"Submission saved — {len(sub_out)} rows")
display(sub_out.head())